In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [39]:
data = pd.read_csv('Churn_Modelling.csv') 
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

# Label encode gender
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

# One-hot encode geography
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(data[['Geography']])
encoded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(['Geography']))

# Remove original Geography and add encoded columns
data.drop('Geography', axis=1, inplace=True)
data = pd.concat([data, encoded_df], axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [40]:
# ---------------- Save the encoders ----------------

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(le, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(ohe, file)

In [41]:
# spitting for x and y
X = data.drop('Exited', axis=1)
Y = data['Exited']

# splitting the data into train and test
X_train,X_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


In [42]:
import tensorflow as tf
import keras
from keras import layers
import datetime

In [43]:
# Buid our ANN model
model = keras.Sequential(
    [
         layers.Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
         layers.Dense(32,activation='relu'),
         layers.Dense(1,activation='sigmoid')
    ]
)   

/Users/mushamsushanth/Documents/ann_classification/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [44]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [46]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

In [47]:
# early stopping
early_stopping_callbaack = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience = 10,
    restore_best_weights=True
 )

In [48]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_test, y_test), 
    callbacks=[tensorboard_callback, early_stopping_callbaack]  
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8109 - loss: 0.4377 - val_accuracy: 0.8385 - val_loss: 0.3832
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 967us/step - accuracy: 0.8460 - loss: 0.3762 - val_accuracy: 0.8545 - val_loss: 0.3530
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - accuracy: 0.8525 - loss: 0.3534 - val_accuracy: 0.8560 - val_loss: 0.3470
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 919us/step - accuracy: 0.8585 - loss: 0.3436 - val_accuracy: 0.8575 - val_loss: 0.3451
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 890us/step - accuracy: 0.8606 - loss: 0.3395 - val_accuracy: 0.8610 - val_loss: 0.3394
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8614 - loss: 0.3347 - val_accuracy: 0.8615 - val_loss: 0.3410
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8627 - loss: 0.3320 - val_accuracy: 0.8605 - val_loss: 0.3408
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8641 - loss: 0.3291 - 

In [49]:
model.save('model.h5')

In [50]:
# load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [51]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 26819), started 0:06:07 ago. (Use '!kill 26819' to kill it.)